# FlashRAG Naive RAG — Evaluation on PubMed Summary QA

Evaluates [FlashRAG](https://github.com/RUC-NLPIR/FlashRAG) (WWW 2025) in **Naive RAG** mode on the first 50 questions  
of `pubmed_summary_qa.csv`, using the same metrics as NeuroRAG and MedRAG.

**Pipeline:**
1. **Corpus** — PubMed abstracts fetched via Entrez API (top-20 per question, deduplicated)
2. **Retriever** — BM25 (`bm25s` backend, no Java/GPU required)
3. **Generator** — `gpt-4o-mini` via OpenRouter (single call, no fusing or reranking)

**Why this is a fair baseline:** FlashRAG Naive RAG is a single-stage pipeline published in a  
peer-reviewed academic paper (WWW 2025). It lacks NeuroRAG's multi-source retrieval,  
multi-model fusing, LLM reranking, and contextual compression.

## 1. Install dependencies

In [1]:
!pip install wavedrom --use-pep517
!pip install flashrag-dev --pre
!pip install bm25s

## 2. Imports

In [2]:
import sys

sys.path.append('..')
sys.path.append('../neurorag')

import os
import json
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import httpx
import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass

from metrics import (
    embeddings_cosine_sim_metric,
    bleu_metric,
    rogue_l_metric,
    rogue_1_metric,
    factscore_metric,
    bert_score_metric,
)

import warnings
warnings.filterwarnings('ignore')

## 3. Environment variables

In [3]:
load_dotenv()

for key in ['OPENROUTER_API_KEY', 'ENTREZ_EMAIL']:
    if not os.getenv(key):
        os.environ[key] = getpass(key)

OPENROUTER_KEY = os.environ['OPENROUTER_API_KEY']
ENTREZ_EMAIL   = os.environ['ENTREZ_EMAIL']

ANSWER_STYLE = (
    'Answer in 2-3 sentences (30-50 words). '
    'State the key fact first, then supporting detail. '
    'Write like a PubMed abstract sentence.'
)

print('Environment ready.')

Environment ready.


## 4. Load dataset

In [4]:
N_QUESTIONS = 50
df = pd.read_csv('../datasets/pubmed_summary_qa.csv').head(N_QUESTIONS)
questions        = df['question'].tolist()
expected_answers = df['answer'].tolist()

print(f'Loaded {len(questions)} questions.')
df.head()

Loaded 50 questions.


,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...


## 5. Build PubMed corpus

Fetch the top-20 PubMed abstracts for each question, deduplicate, and save as a  
`corpus.jsonl` file in FlashRAG format: `{"id": "<pmid>", "contents": "<title + abstract>"}`.  
Results are cached so the cell can be re-run without extra API calls.

In [5]:
CORPUS_FILE = Path('flashrag_corpus.jsonl')
CORPUS_CACHE_FILE = Path('flashrag_corpus_cache.json')
TOP_K_CORPUS = 20  # abstracts per question

ENTREZ_SEARCH = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
ENTREZ_FETCH  = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'


def _search_pmids(query: str, top_k: int) -> list[str]:
    r = httpx.get(ENTREZ_SEARCH, params={
        'db': 'pubmed', 'term': query, 'retmax': top_k,
        'retmode': 'json', 'email': ENTREZ_EMAIL,
    }, timeout=15)
    r.raise_for_status()
    return r.json().get('esearchresult', {}).get('idlist', [])


def _fetch_abstracts(pmids: list[str]) -> dict[str, str]:
    """Returns {pmid: 'Title: ... Abstract: ...'}"""
    if not pmids:
        return {}
    r = httpx.get(ENTREZ_FETCH, params={
        'db': 'pubmed', 'id': ','.join(pmids),
        'retmode': 'xml', 'rettype': 'abstract', 'email': ENTREZ_EMAIL,
    }, timeout=30)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    result = {}
    for article in root.findall('.//PubmedArticle'):
        pmid_el = article.find('.//PMID')
        if pmid_el is None:
            continue
        pmid = pmid_el.text or ''
        title_el = article.find('.//ArticleTitle')
        title = (title_el.text or '').strip() if title_el is not None else ''
        abstract = ' '.join(
            (el.text or '').strip()
            for el in article.findall('.//AbstractText') if el.text
        )
        if abstract:
            result[pmid] = f'Title: {title}\nAbstract: {abstract}'
    return result


# Load previously fetched corpus cache
if CORPUS_CACHE_FILE.exists():
    with open(CORPUS_CACHE_FILE) as f:
        corpus_cache: dict[str, str] = json.load(f)  # {pmid: text}
else:
    corpus_cache = {}

print('Fetching PubMed abstracts...')
for question in tqdm(questions):
    try:
        pmids = _search_pmids(question, top_k=TOP_K_CORPUS)
        new_pmids = [p for p in pmids if p not in corpus_cache]
        if new_pmids:
            fetched = _fetch_abstracts(new_pmids)
            corpus_cache.update(fetched)
        time.sleep(0.35)  # PubMed rate limit: 3 req/s
    except Exception as e:
        print(f'  Warning: {e}')

# Persist
with open(CORPUS_CACHE_FILE, 'w') as f:
    json.dump(corpus_cache, f, ensure_ascii=False)

# Write FlashRAG corpus JSONL
with open(CORPUS_FILE, 'w') as f:
    for pmid, text in corpus_cache.items():
        f.write(json.dumps({'id': pmid, 'contents': text}) + '\n')

print(f'Corpus: {len(corpus_cache)} unique abstracts → {CORPUS_FILE}')

Fetching PubMed abstracts...


100%|██████████| 50/50 [00:52<00:00,  1.05s/it]

Corpus: 490 unique abstracts → flashrag_corpus.jsonl


## 6. Build BM25 index

Uses FlashRAG's `index_builder` with the `bm25s` backend (no Java or GPU required).

In [6]:
import subprocess

INDEX_DIR = Path('flashrag_index')
# The index_builder creates files inside a 'bm25/' subdirectory
_index_marker = INDEX_DIR / 'bm25' / 'params.index.json'

if not _index_marker.exists():
    print('Building BM25 index...')
    result = subprocess.run(
        [
            sys.executable, '-m', 'flashrag.retriever.index_builder',
            '--retrieval_method', 'bm25',
            '--corpus_path', str(CORPUS_FILE),
            '--bm25_backend', 'bm25s',
            '--save_dir', str(INDEX_DIR),
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError('Index build failed')
    print('Index built successfully.')
else:
    print(f'Index already exists at {_index_marker.parent.resolve()}')


Index already exists at /Users/vladimirskvortsov/Projects/neurorag/notebooks/flashrag_index/bm25


## 7. Prepare FlashRAG dataset

FlashRAG expects a `test.jsonl` file with fields `id`, `question`, and `golden_answers`.

In [7]:
DATA_DIR = Path('flashrag_data')
DATA_DIR.mkdir(exist_ok=True)

test_file = DATA_DIR / 'test.jsonl'
with open(test_file, 'w') as f:
    for i, (q, a) in enumerate(zip(questions, expected_answers)):
        f.write(json.dumps({'id': str(i), 'question': q, 'golden_answers': [a]}) + '\n')

print(f'Wrote {len(questions)} questions to {test_file}')

Wrote 50 questions to flashrag_data/test.jsonl


## 8. Configure and run FlashRAG SequentialPipeline

**Naive RAG** = BM25 retrieval → single LLM call (no reranking, no fusing).

In [8]:
from flashrag.config import Config
from flashrag.utils import get_dataset
from flashrag.pipeline import SequentialPipeline
from flashrag.prompt import PromptTemplate

# Re-define paths here so this cell is self-contained
INDEX_DIR = Path('flashrag_index')
BM25_INDEX_DIR = INDEX_DIR / 'bm25'

# Sanity-check — fail fast with a clear message
if not (BM25_INDEX_DIR / 'params.index.json').exists():
    raise FileNotFoundError(
        f'BM25 index not found at {BM25_INDEX_DIR.resolve()}. '
        'Re-run cell 6 (Build BM25 index) first.'
    )

config_dict = {
    # Paths
    'data_dir':    str(DATA_DIR),
    'save_dir':    'flashrag_output/',
    'corpus_path': str(CORPUS_FILE.resolve()),
    'dataset_name': 'pubmed_qa',
    'split': ['test'],

    # Retriever — BM25 (bm25s backend, files live in INDEX_DIR/bm25/)
    'retrieval_method': 'bm25',
    'bm25_backend':     'bm25s',
    'index_path':       str(BM25_INDEX_DIR.resolve()),
    'retrieval_topk':   10,
    'silent_retrieval': True,

    # Generator — OpenAI-compatible (-> OpenRouter)
    'framework':       'openai',
    'generator_model': 'openai/gpt-4o-mini',
    'openai_setting': {
        'api_key':  OPENROUTER_KEY,
        'base_url': 'https://openrouter.ai/api/v1',
    },
    'generation_params': {
        'max_tokens':   200,
        'temperature':  0,
    },
    'generator_max_input_len': 4096,

    # Evaluation — we compute our own metrics afterwards
    'metrics': [],
    'save_intermediate_data': True,
    'save_metric_score': False,
    'save_note': 'flashrag_naive',
}

config = Config(config_dict=config_dict)

# Custom prompt matching our ANSWER_STYLE
prompt_template = PromptTemplate(
    config,
    system_prompt=(
        f'You are a biomedical expert. OUTPUT RULE: {ANSWER_STYLE} '
        'Answer the question using only the provided documents. '
        'Be direct and factual - no preamble, no caveats.\n\n'
        'Documents:\n{reference}'
    ),
    user_prompt='Question: {question}\nAnswer:',
)

all_split = get_dataset(config)
test_data = all_split['test']

pipeline = SequentialPipeline(config, prompt_template=prompt_template)

print(f'Pipeline ready. Running on {len(test_data)} questions...')
output_dataset = pipeline.run(test_data, do_eval=False)
print('Done.')


Generating train split: 0 examples [00:00, ? examples/s]

TypeError: object of type 'NoneType' has no len()

## 9. Extract predicted answers

In [ ]:
predicted_answers = output_dataset.pred

# Sanity check
ref_lens  = [len(a.split()) for a in expected_answers]
pred_lens = [len(a.split()) for a in predicted_answers]

print(f'Reference  avg: {np.mean(ref_lens):.1f}w')
print(f'Predicted  avg: {np.mean(pred_lens):.1f}w')
print()

for q, ref, pred in list(zip(questions, expected_answers, predicted_answers))[:3]:
    print(f'Q:    {q[:70]}')
    print(f'REF:  {ref[:130]}')
    print(f'PRED: {pred[:130]}')
    print()

## 10. Save results

In [ ]:
results_df = pd.DataFrame({
    'question':         questions,
    'expected_answer':  expected_answers,
    'flashrag_answer':  predicted_answers,
})
results_df.to_csv('flashrag_results.csv', index=False)

# Also cache for re-use
cache = dict(zip(questions, predicted_answers))
with open('flashrag_cache.json', 'w') as f:
    json.dump(cache, f, indent=2, ensure_ascii=False)

print('Saved flashrag_results.csv and flashrag_cache.json')
results_df.head()

## 11. Compute metrics

In [ ]:
metrics = {
    'cos_score':       round(float(embeddings_cosine_sim_metric(expected_answers, predicted_answers)), 4),
    'bleu_score':      round(float(bleu_metric(expected_answers, predicted_answers)), 4),
    'rouge_1_score':   round(float(rogue_1_metric(expected_answers, predicted_answers)), 4),
    'rouge_l_score':   round(float(rogue_l_metric(expected_answers, predicted_answers)), 4),
    'factscore_score': round(float(factscore_metric(expected_answers, predicted_answers)), 4),
    'bert_score':      round(float(bert_score_metric(expected_answers, predicted_answers)), 4),
}

print('=== FlashRAG Naive RAG Metrics ===')
for name, value in metrics.items():
    print(f'{name:<20} {value:.4f}')

## 12. Compare all systems

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Update with your latest results ─────────────────────────────────────────
neurorag_scores = {
    'cos_score': 0.7615, 'bleu_score': 0.0729, 'rouge_1_score': 0.3677,
    'rouge_l_score': 0.3010, 'factscore_score': 0.1643, 'bert_score': 0.4248,
}
medrag_scores = {
    'cos_score': 0.7794, 'bleu_score': 0.0917, 'rouge_1_score': 0.4111,
    'rouge_l_score': 0.3291, 'factscore_score': 0.3040, 'bert_score': 0.4241,
}
vanilla_pubmed_scores = {
    'cos_score': 0.0, 'bleu_score': 0.0, 'rouge_1_score': 0.0,
    'rouge_l_score': 0.0, 'factscore_score': 0.0, 'bert_score': 0.0,
}  # fill in after running pubmedrag-evaluation.ipynb
# ────────────────────────────────────────────────────────────────────────────

metric_names = list(metrics.keys())
labels       = [m.replace('_score', '') for m in metric_names]

systems = {
    'NeuroRAG':               ([neurorag_scores[m]        for m in metric_names], 'steelblue'),
    'FlashRAG Naive (BM25)':  ([metrics[m]                for m in metric_names], 'mediumpurple'),
    'MedRAG (BM25+Textbooks)':([medrag_scores[m]          for m in metric_names], 'darkorange'),
}

x     = np.arange(len(metric_names))
width = 0.25
fig, ax = plt.subplots(figsize=(13, 5))

for i, (name, (vals, color)) in enumerate(systems.items()):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=name, color=color, alpha=0.87)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=7.5,
        )

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('RAG System Comparison — PubMed Summary QA (n=50)')
ax.legend()
ax.set_ylim(0, 1)
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('flashrag_vs_neurorag.png', dpi=150)
plt.show()

# Summary table
comparison = pd.DataFrame(
    {name: vals for name, (vals, _) in systems.items()},
    index=labels,
)
comparison['NeuroRAG vs FlashRAG'] = (
    comparison['NeuroRAG'] - comparison['FlashRAG Naive (BM25)']
).round(4)
comparison